# Notebook 06 — Export Data for Website Visualizations

This notebook:
1. Loads the clustered artifact data
2. Aggregates city-level statistics
3. Exports JSON files that the website's JavaScript reads to render
   the Semantic Space Explorer and Geographic View

Output files (written to `website/assets/data/`):
- `artifacts.json` — one record per artifact with metadata
- `projections.json` — UMAP coordinates (already generated by notebook 05)
- `cities.json` — city-level counts by artifact type

**Before running:** complete notebook 05.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import shutil
import pandas as pd
from pathlib import Path

PROCESSED = Path('../data/processed')
EMBEDDINGS = Path('../data/embeddings')
WEB_DATA = Path('../website/assets/data')
WEB_DATA.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- 1. Load combined artifact data ---

df = pd.read_csv(PROCESSED / 'all_artifacts.csv')
print(f'{len(df)} artifacts loaded')

In [ ]:
# --- 2. Export artifacts.json ---
# Keep only the fields the website needs to stay lightweight.

web_cols = ['id', 'type', 'title', 'year', 'city', 'country',
            'case_study_flag', 'cluster_label']
artifacts = df[web_cols].where(pd.notnull(df[web_cols]), None).to_dict(orient='records')

with open(WEB_DATA / 'artifacts.json', 'w') as f:
    json.dump(artifacts, f)
print(f'Wrote artifacts.json ({len(artifacts)} records)')

In [ ]:
# --- 3. Copy projections.json to website ---

shutil.copy(EMBEDDINGS / 'projections.json', WEB_DATA / 'projections.json')
print('Copied projections.json')

In [ ]:
# --- 4. Build and export cities.json ---

city_df = df.dropna(subset=['city', 'lat', 'lon'])

cities = (
    city_df.groupby(['city', 'country', 'lat', 'lon'])
    .apply(lambda g: pd.Series({
        'count_papers': (g['type'] == 'paper').sum(),
        'count_patents': (g['type'] == 'patent').sum(),
        'count_projects': (g['type'] == 'project').sum(),
        'count_parts': (g['type'] == 'part').sum(),
        'count_carbon_capture': g['case_study_flag'].sum(),
    }))
    .reset_index()
    .to_dict(orient='records')
)

with open(WEB_DATA / 'cities.json', 'w') as f:
    json.dump(cities, f)
print(f'Wrote cities.json ({len(cities)} cities)')